# ML Assignment 2 — Model Training
**Dataset:** Breast Cancer Wisconsin (Diagnostic) — 30 features, 569 instances, binary classification.

Trains Logistic Regression, Decision Tree, kNN, Naive Bayes, and Random Forest; computes Accuracy, AUC, Precision, Recall, F1, and MCC for each; saves models for the Streamlit app.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

RANDOM_STATE = 42


## 1. Load and split the dataset

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target  # 0 = malignant, 1 = benign

print(f"Dataset shape: {X.shape}, classes: {sorted(y.unique())}")
assert X.shape[1] >= 12 and X.shape[0] >= 500, "Dataset does not meet assignment minimums"

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Dataset shape: (569, 30), classes: [0, 1]


## 2. Save the test split (used later by the Streamlit app's CSV upload)

In [ ]:
test_df = X_test.copy()
test_df["target"] = y_test.values
test_df.to_csv("test_data.csv", index=False)
print("Saved test_data.csv with shape", test_df.shape)


Saved test_data.csv with shape (114, 31)


## 3. Define the 5 classification models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}


## 4. Train each model, evaluate, and save

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    metrics = {
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_proba), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
    }
    results.append(metrics)
    print(metrics)

    fname = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    joblib.dump(model, f"{fname}.pkl")

joblib.dump(scaler, "scaler.pkl")
with open("feature_names.json", "w") as f:
    json.dump(list(X.columns), f)


{'ML Model Name': 'Logistic Regression', 'Accuracy': 0.9825, 'AUC': 0.9954, 'Precision': 0.9861, 'Recall': 0.9861, 'F1': 0.9861, 'MCC': 0.9623}
{'ML Model Name': 'Decision Tree', 'Accuracy': 0.9123, 'AUC': 0.9157, 'Precision': 0.9559, 'Recall': 0.9028, 'F1': 0.9286, 'MCC': 0.8174}
{'ML Model Name': 'kNN', 'Accuracy': 0.9561, 'AUC': 0.9788, 'Precision': 0.9589, 'Recall': 0.9722, 'F1': 0.9655, 'MCC': 0.9054}
{'ML Model Name': 'Naive Bayes', 'Accuracy': 0.9298, 'AUC': 0.9868, 'Precision': 0.9444, 'Recall': 0.9444, 'F1': 0.9444, 'MCC': 0.8492}
{'ML Model Name': 'Random Forest (Ensemble)', 'Accuracy': 0.9561, 'AUC': 0.9932, 'Precision': 0.9589, 'Recall': 0.9722, 'F1': 0.9655, 'MCC': 0.9054}


## 5. Comparison table

In [ ]:
metrics_df = pd.DataFrame(results)
metrics_df.to_csv("metrics_summary.csv", index=False)
print(metrics_df.to_string(index=False))


            ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
     Logistic Regression    0.9825 0.9954     0.9861  0.9861 0.9861 0.9623
           Decision Tree    0.9123 0.9157     0.9559  0.9028 0.9286 0.8174
                     kNN    0.9561 0.9788     0.9589  0.9722 0.9655 0.9054
             Naive Bayes    0.9298 0.9868     0.9444  0.9444 0.9444 0.8492
Random Forest (Ensemble)    0.9561 0.9932     0.9589  0.9722 0.9655 0.9054
